# Inferencia RF en curva unica + gate a allesfitter (MCMC)

Esta libreta toma una curva salida de fotometria (como en la libreta 4), la adapta al formato del modelo RF, ejecuta clasificacion y, si sale positiva, prepara automaticamente los inputs para allesfitter (`data.csv`, `settings.csv`, `params.csv`).

## 1. Imports y contexto de proyecto

In [17]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from classifier.code.inference import load_model_bundle, extract_features_from_csv, predict_single

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM


## 2. Configuracion de entrada (curva detrended_masked)

In [ ]:
# Modelo RF elegido para produccion
RF_ARTIFACTS = PROJECT_ROOT / "src/classifier/artifacts/rf_final"

# Salida esperada de la libreta 4 (fotometria)
PHOTOMETRY_CURVE_PATH = PROJECT_ROOT / "data/photometry/differential_light_curve_detrended_masked.csv"

# Alternativa si prefieres leer el export dentro de src/photometry
# PHOTOMETRY_CURVE_PATH = PROJECT_ROOT / "src/photometry/differential_light_curve_detrended_masked.csv"

# Objetivo actual
TARGET_NAME = "WASP-2b"

# Instrumento fotometrico por defecto para allesfitter
DEFAULT_INSTRUMENT = "data"

# Si True, intenta autocompletar parametros desde LITERATURE_PATH
AUTO_CONFIG_FROM_LITERATURE = True

# Rutas para pipeline de curva unica
SINGLE_DIR = PROJECT_ROOT / "src/classifier/data/single_inference"
SINGLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INPUT_CURVE = SINGLE_DIR / "curve_for_rf.csv"

# Literatura (opcional): CSV con columnas recomendadas
# target,param,value,sigma,source
LITERATURE_PATH = PROJECT_ROOT / "data/literature/transit_parameters_literature.csv"

print(f"RF artifacts: {RF_ARTIFACTS} | exists={RF_ARTIFACTS.exists()}")
print(f"Curva fotometria: {PHOTOMETRY_CURVE_PATH} | exists={PHOTOMETRY_CURVE_PATH.exists()}")
print(f"Curva para RF: {MODEL_INPUT_CURVE}")
print(f"Target actual: {TARGET_NAME}")
print(f"Literatura: {LITERATURE_PATH} | exists={LITERATURE_PATH.exists()}")

RF artifacts: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\classifier\artifacts\rf_final | exists=True
Curva fotometria: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\data\photometry\local_pos_1_00254.csv | exists=True
Curva para RF: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\classifier\data\single_inference\curve_for_rf.csv
Target actual: WASP-2b
Literatura: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\data\literature\transit_parameters_literature.csv | exists=False


## 3. Adaptar curva de fotometria al formato del modelo RF

In [20]:
if not PHOTOMETRY_CURVE_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro la curva de fotometria: {PHOTOMETRY_CURVE_PATH}\n"
        "Ejecuta la libreta 4 y ajusta PHOTOMETRY_CURVE_PATH."
    )

lc = pd.read_csv(PHOTOMETRY_CURVE_PATH)

time_col = "time_jd" if "time_jd" in lc.columns and lc["time_jd"].notna().any() else "frame_index"

# Compatibilidad: si existe detrended_masked la usamos; si no, usamos detrended_flux.
if "detrended_masked" in lc.columns:
    flux_col_in = "detrended_masked"
elif "detrended_flux" in lc.columns:
    flux_col_in = "detrended_flux"
else:
    raise ValueError("No se encontro columna de flujo detrended (detrended_masked o detrended_flux).")

rf_curve = lc[[time_col, flux_col_in]].copy()
rf_curve = rf_curve.rename(columns={time_col: "time_jd", flux_col_in: "detrended_flux"})
rf_curve = rf_curve.replace([np.inf, -np.inf], np.nan).dropna(subset=["time_jd", "detrended_flux"])

if len(rf_curve) < 20:
    raise ValueError(f"La curva tiene {len(rf_curve)} puntos validos; se requieren al menos 20.")

rf_curve.to_csv(MODEL_INPUT_CURVE, index=False)
print(f"Curva adaptada guardada en: {MODEL_INPUT_CURVE}")
print(f"Puntos validos: {len(rf_curve)}")
display(rf_curve.head(10))

Curva adaptada guardada en: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\classifier\data\single_inference\curve_for_rf.csv
Puntos validos: 260


,time_jd,detrended_flux
0,2.459283e+06,1.000180
1,2.459283e+06,1.003478
2,2.459283e+06,0.998386
3,2.459283e+06,0.998145
4,2.459283e+06,1.008305
5,2.459283e+06,1.006714
6,2.459283e+06,1.004948
7,2.459283e+06,1.004822
8,2.459283e+06,1.002297
9,2.459283e+06,0.999715


## 4. Inferencia de una curva con Random Forest

In [21]:
bundle_rf = load_model_bundle(RF_ARTIFACTS, name="rf")
features = extract_features_from_csv(MODEL_INPUT_CURVE, flux_column="detrended_flux", time_column="time_jd")
pred = predict_single(bundle_rf, features)

prediction_row = {
    "target": TARGET_NAME,
    "curve_path": str(MODEL_INPUT_CURVE),
    "model": "rf",
    "threshold": pred["threshold"],
    "prob_positive": pred["prob_positive"],
    "label_pred": pred["label_pred"],
}

pred_df = pd.DataFrame([prediction_row])
display(pred_df)

is_positive = int(pred["label_pred"]) == 1
print("Resultado RF: POSITIVA -> pasa a ajuste bayesiano" if is_positive else "Resultado RF: NEGATIVA -> no pasa a ajuste bayesiano")

,target,curve_path,model,threshold,prob_positive,label_pred
0,WASP-2b,C:\Users\Daniel\Desktop\En curso\Máster Astrof...,rf,0.5,0.97675,1


Resultado RF: POSITIVA -> pasa a ajuste bayesiano


## 5. Gate: preparar inputs para allesfitter (MCMC emcee)

In [5]:
ALLESFITTER_ROOT = PROJECT_ROOT / "src/bayesian"
ALLESFITTER_INPUT_DIR = ALLESFITTER_ROOT / "data/input" / TARGET_NAME
ALLESFITTER_INPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CSV_PATH = ALLESFITTER_INPUT_DIR / "data.csv"
SETTINGS_CSV_PATH = ALLESFITTER_INPUT_DIR / "settings.csv"
PARAMS_CSV_PATH = ALLESFITTER_INPUT_DIR / "params.csv"
PARAMS_STAR_CSV_PATH = ALLESFITTER_INPUT_DIR / "params_star.csv"

In [10]:
import numpy as np
import pandas as pd

df = pd.read_csv(ALLESFITTER_INPUT_DIR / "data.csv", comment="#", names=["time", "flux", "flux_err"])

# Seleccionar solo los puntos en tránsito (flujo < umbral)
in_transit = df[df["flux"] < 0.995]

# Centroide ponderado por profundidad
weights = 1 - in_transit["flux"]  # más peso a los puntos más profundos
t0_centroid = np.average(in_transit["time"], weights=weights)
print(f"T0 centroide: {t0_centroid:.7f}")

T0 centroide: 2457581.4127679


In [11]:

# Plantilla manual por objetivo (fallback).
# Puedes añadir aqui todos tus objetivos y dejar que TARGET_NAME seleccione el adecuado.
PLANET_CONFIGS = {
    "WASP-2b": {
        "companion": "b",
        "instrument": DEFAULT_INSTRUMENT,
        "rr": 0.1326,
        "rsuma": 0.140173267,
        "cosi": 0.09045876,
        "epoch": 2457581.4127679,
        "period": 2.152221956,
        "host_ldc_q1": 0.292,
        "host_ldc_q2": 0.213,
        # Params estelares para params_star.csv
        "R_star": 0.821,
        "R_star_lerr": 0.05,
        "R_star_uerr": 0.05,
        "M_star": 0.843,
        "M_star_lerr": 0.05,
        "M_star_uerr": 0.05,
        "Teff_star": 5170.0,
        "Teff_star_lerr": 60.0,
        "Teff_star_uerr": 60.0,
    },
}


def _infer_companion_from_target(target_name: str) -> str:
    t = str(target_name).strip()
    if len(t) > 0 and t[-1].isalpha():
        return t[-1].lower()
    return "b"


def resolve_planet_config(
    target_name: str,
    default_instrument: str,
    literature_path: Path,
    auto_from_literature: bool,
    manual_configs: dict[str, dict[str, object]],
) -> tuple[dict[str, object], str]:
    t = str(target_name).strip()

    # 1) Si existe config manual exacta, usarla
    if t in manual_configs:
        cfg = dict(manual_configs[t])
        cfg.setdefault("companion", _infer_companion_from_target(t))
        cfg.setdefault("instrument", default_instrument)
        return cfg, "manual"

    # 2) Intentar autoconfig desde literatura
    if auto_from_literature and literature_path.exists():
        lit_df = pd.read_csv(literature_path)
        required_cols = {"target", "param", "value"}
        if required_cols.issubset(set(lit_df.columns)):
            m = lit_df[lit_df["target"].astype(str).str.lower() == t.lower()].copy()
            if len(m) > 0:
                alias = {
                    "rr": "rr",
                    "rprs": "rr",
                    "rp_rs": "rr",
                    "rp_over_rs": "rr",
                    "rsuma": "rsuma",
                    "rstar_plus_rp_over_a": "rsuma",
                    "cosi": "cosi",
                    "cos_i": "cosi",
                    "epoch": "epoch",
                    "t0": "epoch",
                    "tc": "epoch",
                    "period": "period",
                    "p": "period",
                }

                cfg_auto: dict[str, object] = {
                    "companion": _infer_companion_from_target(t),
                    "instrument": default_instrument,
                }

                for row in m.itertuples(index=False):
                    p = str(getattr(row, "param")).strip().lower()
                    if p in alias:
                        try:
                            cfg_auto[alias[p]] = float(getattr(row, "value"))
                        except Exception:
                            pass

                needed = ["rr", "rsuma", "cosi", "epoch", "period"]
                if all(k in cfg_auto for k in needed):
                    return cfg_auto, "literature"

    raise KeyError(
        f"No hay configuracion allesfitter valida para {t}. "
        "O bien añade una entrada en PLANET_CONFIGS, "
        "o bien completa LITERATURE_PATH con params: rr, rsuma, cosi, epoch, period."
    )


if True:
    cfg, cfg_source = resolve_planet_config(
        target_name=TARGET_NAME,
        default_instrument=DEFAULT_INSTRUMENT,
        literature_path=LITERATURE_PATH,
        auto_from_literature=AUTO_CONFIG_FROM_LITERATURE,
        manual_configs=PLANET_CONFIGS,
    )

    companion = str(cfg["companion"])
    instrument = str(cfg["instrument"])

    # 1) data.csv en formato allesfitter: time, flux, flux_err
    #    Escribimos cabecera comentada para que allesfitter la ignore.
    data_df = rf_curve[["time_jd", "detrended_flux"]].copy()
    data_df = data_df.rename(columns={"time_jd": "time", "detrended_flux": "flux"})
    data_df = data_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["time", "flux"])
    data_df = data_df.drop_duplicates(subset=["time"], keep="first").sort_values("time").reset_index(drop=True)

    if len(data_df) < 20:
        raise ValueError(f"Tras limpieza quedaron {len(data_df)} puntos; se requieren al menos 20.")

    # Estimacion robusta de error fotometrico escalar para todas las filas.
    flux = data_df["flux"].to_numpy(dtype=float)
    mad = np.median(np.abs(flux - np.median(flux)))
    sigma_mad = 1.4826 * mad if np.isfinite(mad) else np.nan
    if len(flux) > 2:
        sigma_diff = np.std(np.diff(flux), ddof=1) / np.sqrt(2.0)
    else:
        sigma_diff = np.nan
    sigma_candidates = [v for v in [sigma_mad, sigma_diff] if np.isfinite(v) and v > 0]
    flux_err_value = float(np.median(sigma_candidates)) if sigma_candidates else 1e-4
    data_df["flux_err"] = flux_err_value

    with DATA_CSV_PATH.open("w", encoding="utf-8", newline="") as f:
        f.write("#time,flux,flux_err\n")
        data_df.to_csv(f, index=False, header=False)

    # 2) settings.csv (MCMC emcee, 5000 pasos por defecto)
    settings_lines = [
        "#name,value",
        "###############################################################################,",
        "# General settings,",
        "###############################################################################,",
        f"companions_phot,{companion}",
        "companions_rv,",
        f"inst_phot,{instrument}",
        "inst_rv,",
        "###############################################################################,",
        "# Fit performance settings,",
        "###############################################################################,",
        "multiprocess,False",
        "multiprocess_cores,1",
        "fast_fit,True",
        "fast_fit_width,0.3333333333333333",
        "shift_epoch,True",
        "inst_for_b_epoch,all",
        "###############################################################################,",
        "# MCMC settings,",
        "###############################################################################,",
        "mcmc_nwalkers,100",
        "mcmc_total_steps,5000",
        "mcmc_burn_steps,1000",
        "mcmc_thin_by,10",
        "###############################################################################,",
        "# Nested Sampling settings,",
        "###############################################################################,",
        "ns_modus,dynamic",
        "ns_nlive,500",
        "ns_bound,single",
        "ns_sample,rwalk",
        "ns_tol,0.01",
        "###############################################################################,",
        "# Limb darkening law per object and instrument,",
        "###############################################################################,",
        f"host_ld_law_{instrument},quad",
        "###############################################################################,",
        "# Baseline settings per instrument,",
        "###############################################################################,",
        f"baseline_flux_{instrument},sample_offset",
        "###############################################################################,",
        "# Error settings per instrument,",
        "###############################################################################,",
        f"error_flux_{instrument},sample",
        "###############################################################################,",
        "# Stellar grid per object and instrument,",
        "###############################################################################,",
        f"host_grid_{instrument},very_sparse",
        f"{companion}_grid_{instrument},very_sparse",
    ]
    SETTINGS_CSV_PATH.write_text("\n".join(settings_lines) + "\n", encoding="utf-8")

    # 3) params.csv (plantilla tipo allesfitter basada en el objetivo)
    rr = float(cfg["rr"])
    rsuma = float(cfg["rsuma"])
    cosi = float(cfg["cosi"])
    epoch = float(cfg["epoch"])
    period = float(cfg["period"])
    host_ldc_q1 = float(cfg.get("host_ldc_q1", 0.5))
    host_ldc_q2 = float(cfg.get("host_ldc_q2", 0.5))

    params_lines = [
        "#name,value,fit,bounds,label,unit,truth",
        f"#{companion} astrophysical params,,,,,,",
        f"{companion}_rr,{rr},1,uniform {max(0.001, rr * 0.5):.6f} {rr * 1.5:.6f},$R_{companion} / R_\\star$,,{rr}",
        f"{companion}_rsuma,{rsuma},1,uniform {max(0.001, rsuma * 0.5):.6f} {rsuma * 1.5:.6f},$(R_\\star + R_{companion}) / a_{companion}$,,{rsuma}",
        f"{companion}_cosi,{cosi},1,uniform 0.0 0.2,$\\cos{{i_{companion}}}$,,{cosi}",
        f"{companion}_epoch,{epoch},1,uniform {epoch - 0.05:.6f} {epoch + 0.05:.6f},$T_{{0;{companion}}}$,BJD,{epoch}",
        f"{companion}_period,{period},1,uniform {period - 0.05:.6f} {period + 0.05:.6f},$P_{companion}$,days,{period}",
        "#limb darkening coefficients per instrument,,,,,,",
        f"host_ldc_q1_{instrument},{host_ldc_q1},1,uniform 0.0 1.0,$q_{{1; \\mathrm{{{instrument}}}}}$,,{host_ldc_q1}",
        f"host_ldc_q2_{instrument},{host_ldc_q2},1,uniform 0.0 1.0,$q_{{2; \\mathrm{{{instrument}}}}}$,,{host_ldc_q2}",
        "#errors per instrument,,,,,,",
        f"ln_err_flux_{instrument},-7,1,uniform -8 -5,$\\log{{\\sigma_\\mathrm{{{instrument}}}}}$,$\\log{{ \\mathrm{{rel. flux.}} }}$,",
        "#baseline per instrument,,,,,,",
        f"baseline_offset_flux_{instrument},0,1,uniform -0.003 0.003,$\\mathrm{{gp: \\log{{\\sigma}} ({instrument})}}$,,",
    ]
    PARAMS_CSV_PATH.write_text("\n".join(params_lines) + "\n", encoding="utf-8")

    # 4) params_star.csv con formato allesfitter
    r_star = float(cfg.get("R_star", 1.0))
    r_star_lerr = float(cfg.get("R_star_lerr", 0.05))
    r_star_uerr = float(cfg.get("R_star_uerr", 0.05))
    m_star = float(cfg.get("M_star", 1.0))
    m_star_lerr = float(cfg.get("M_star_lerr", 0.05))
    m_star_uerr = float(cfg.get("M_star_uerr", 0.05))
    teff_star = float(cfg.get("Teff_star", 5777.0))
    teff_star_lerr = float(cfg.get("Teff_star_lerr", 100.0))
    teff_star_uerr = float(cfg.get("Teff_star_uerr", 100.0))

    params_star_lines = [
        "#R_star,R_star_lerr,R_star_uerr,M_star,M_star_lerr,M_star_uerr,Teff_star,Teff_star_lerr,Teff_star_uerr",
        "#R_sun,R_sun,R_sun,M_sun,M_sun,M_sun,K,K,K",
        f"{r_star},{r_star_lerr},{r_star_uerr},{m_star},{m_star_lerr},{m_star_uerr},{teff_star},{teff_star_lerr},{teff_star_uerr}",
    ]
    PARAMS_STAR_CSV_PATH.write_text("\n".join(params_star_lines) + "\n", encoding="utf-8")

    print(f"RF positivo. Inputs allesfitter preparados en: {ALLESFITTER_INPUT_DIR}")
    print(f"  - config source: {cfg_source}")
    print(f"  - data.csv      : {DATA_CSV_PATH} (n={len(data_df)}, flux_err={flux_err_value:.3e})")
    print(f"  - settings.csv  : {SETTINGS_CSV_PATH}")
    print(f"  - params.csv    : {PARAMS_CSV_PATH}")
    print(f"  - params_star.csv: {PARAMS_STAR_CSV_PATH}")
else:
    print("No se generan archivos de allesfitter porque la curva no fue clasificada como positiva por RF.")

RF positivo. Inputs allesfitter preparados en: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\bayesian\data\input\WASP-2b
  - config source: manual
  - data.csv      : C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\bayesian\data\input\WASP-2b\data.csv (n=134, flux_err=5.469e-03)
  - settings.csv  : C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\bayesian\data\input\WASP-2b\settings.csv
  - params.csv    : C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\bayesian\data\input\WASP-2b\params.csv
  - params_star.csv: C:\Users\Daniel\Desktop\En curso\Máster Astrofísica\TFM\src\bayesian\data\input\WASP-2b\params_star.csv


## 6. Comparacion con literatura (estructura lista)

In [15]:
import numpy as np
if not hasattr(np, "float"):   np.float   = float
if not hasattr(np, "int"):     np.int     = int
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "bool"):    np.bool    = bool
if not hasattr(np, "object"):  np.object  = object
if not hasattr(np, "str"):     np.str     = str

import warnings
import multiprocessing as mp

# --- PARCHES DE COMPATIBILIDAD (NumPy + Windows) ---
# 1) Clases que algunas versiones de allesfitter esperan de NumPy
if not hasattr(np, "VisibleDeprecationWarning"):
    class DummyVisibleDeprecationWarning(UserWarning):
        pass
    np.VisibleDeprecationWarning = DummyVisibleDeprecationWarning

if not hasattr(np, "RankWarning"):
    class DummyRankWarning(UserWarning):
        pass
    np.RankWarning = DummyRankWarning

# 2) En Windows no existe start method 'fork'.
#    Allesfitter intenta forzarlo al importar mcmc, así que interceptamos esa llamada
_orig_set_start_method = mp.set_start_method

def _safe_set_start_method(method, force=False):
    if method == "fork":
        method = "spawn"
    try:
        return _orig_set_start_method(method, force=force)
    except RuntimeError:
        # Si el contexto ya estaba definido, no rompemos la ejecución
        return None

mp.set_start_method = _safe_set_start_method

# Lanzamos allesfitter con MCMC (emcee)
if True:
    print("\nEjecutando allesfitter con MCMC (emcee)...")
    try:
        import allesfitter
    except Exception as e:
        print(f"No se pudo importar allesfitter: {e}")
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=np.VisibleDeprecationWarning)
            warnings.simplefilter("ignore", category=np.RankWarning)

            try:
                allesfitter.mcmc_fit(ALLESFITTER_INPUT_DIR)
                print("Allesfitter MCMC completado exitosamente.")
            except Exception as e:
                print(f"Error durante el ajuste con allesfitter: {e}")
else:
    print("No se ejecuta allesfitter porque la curva no fue clasificada como positiva por RF.")


Ejecutando allesfitter con MCMC (emcee)...
Filling the Basement


allesfitter version
---------------------
v1.2.10
OrderedDict([('user-given:', ''),
             ('companions_phot', ['b']),
             ('companions_rv', []),
             ('inst_phot', ['data']),
             ('inst_rv', []),
             ('multiprocess', False),
             ('multiprocess_cores', '1'),
             ('fast_fit', 'True'),
             ('fast_fit_width', '0.3333333333333333'),
             ('shift_epoch', True),
             ('inst_for_b_epoch', ['data']),
             ('mcmc_nwalkers', '24'),
             ('mcmc_total_steps', '3000'),
             ('mcmc_burn_steps', '500'),
             ('mcmc_thin_by', '10'),
             ('ns_modus', 'dynamic'),
             ('ns_nlive', '500'),
             ('ns_bound', 'single'),
             ('ns_sample', 'rwalk'),
             ('ns_tol', '0.01'),
             ('host_ld_law_data', 'quad'),
             ('baseline_flux_data', 'sample_offset'),
             ('err

100%|██████████| 3000/3000 [15:52<00:00,  3.15it/s]


Time taken to run 'emcee' on a single core is 0.26 hours

Acceptance fractions:
--------------------------
[0.29666667 0.27666667 0.25333333 0.25333333 0.26666667 0.24666667
 0.27       0.28666667 0.26666667 0.27333333 0.33       0.28666667
 0.30333333 0.29333333 0.28       0.33       0.24       0.23
 0.26       0.28666667 0.33       0.29666667 0.30666667 0.27      ]

Convergence check
-------------------
Total steps:         3000      
Burn steps:          500       
Evaluation steps:    2500                
Evaluation samples:  6000                
Autocorrelation times:
	 parameter                      tau (in steps)       Chain length (in multiples of tau)
	 b_rr                           27.15049446715394    92.07935431984282   
	 b_rsuma                        26.634579706186543   93.86294161868501   
	 b_cosi                         27.12897478174746    92.1523949987972    
	 b_epoch                        22.87848860687326    109.2729525519854   
	 ln_err_flux_data            

In [ ]:
import importlib, traceback

deps = ["ellc", "george", "dynesty", "emcee", "allesfitter"]
for dep in deps:
    try:
        importlib.import_module(dep)
        print(f"✅ {dep} OK")
    except Exception as e:
        print(f"❌ {dep} FALLO: {e}")
        traceback.print_exc()

In [16]:
import numpy as np
if not hasattr(np, "float"):   np.float   = float
if not hasattr(np, "int"):     np.int     = int
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "bool"):    np.bool    = bool
if not hasattr(np, "object"):  np.object  = object
if not hasattr(np, "str"):     np.str     = str

import warnings
import multiprocessing as mp

# --- PARCHES DE COMPATIBILIDAD (NumPy + Windows) ---
# 1) Clases que algunas versiones de allesfitter esperan de NumPy
if not hasattr(np, "VisibleDeprecationWarning"):
    class DummyVisibleDeprecationWarning(UserWarning):
        pass
    np.VisibleDeprecationWarning = DummyVisibleDeprecationWarning

if not hasattr(np, "RankWarning"):
    class DummyRankWarning(UserWarning):
        pass
    np.RankWarning = DummyRankWarning

# 2) En Windows no existe start method 'fork'.
#    Allesfitter intenta forzarlo al importar mcmc, así que interceptamos esa llamada
_orig_set_start_method = mp.set_start_method

def _safe_set_start_method(method, force=False):
    if method == "fork":
        method = "spawn"
    try:
        return _orig_set_start_method(method, force=force)
    except RuntimeError:
        # Si el contexto ya estaba definido, no rompemos la ejecución
        return None

mp.set_start_method = _safe_set_start_method

import allesfitter
allesfitter.mcmc_output(ALLESFITTER_INPUT_DIR)

Filling the Basement


allesfitter version
---------------------
v1.2.10
OrderedDict([('user-given:', ''),
             ('companions_phot', ['b']),
             ('companions_rv', []),
             ('inst_phot', ['data']),
             ('inst_rv', []),
             ('multiprocess', False),
             ('multiprocess_cores', '1'),
             ('fast_fit', 'True'),
             ('fast_fit_width', '0.3333333333333333'),
             ('shift_epoch', True),
             ('inst_for_b_epoch', ['data']),
             ('mcmc_nwalkers', '24'),
             ('mcmc_total_steps', '3000'),
             ('mcmc_burn_steps', '500'),
             ('mcmc_thin_by', '10'),
             ('ns_modus', 'dynamic'),
             ('ns_nlive', '500'),
             ('ns_bound', 'single'),
             ('ns_sample', 'rwalk'),
             ('ns_tol', '0.01'),
             ('host_ld_law_data', 'quad'),
             ('baseline_flux_data', 'sample_offset'),
             ('error_flux_data', 'sample'),
             ('hos

100%|██████████| 5/5 [00:01<00:00,  4.19it/s]


Plotting individual transits for companion b and instrument data...


100%|██████████| 1/1 [00:00<00:00,  3.43it/s]


Deriving eclipse depths (and more) from the model curves for companion b and instrument data...

Saved mcmc_derived_results.csv, mcmc_derived_latex_table.txt, and mcmc_derived_latex_cmd.txt

Saved mcmc_derived_corner.pdf

Performing diagnostic tests on the fit's residuals...

Anderson-Darling Test
---------------------
This tests the null hypothesis that the residuals follows a normal distribution.
Test statistic		 0.4502491018174908
Critical values		 [0.558 0.627 0.748 0.868 1.029]
Significance levels	 [0.15  0.1   0.05  0.025 0.01 ]
Does the null hypotheses hold at a significance level of...
... 0.15 		 True
... 0.1 		 True
... 0.05 		 True
... 0.025 		 True
... 0.01 		 True
The null hypothesis cannot be rejected.
In simple words: your residuals look good.


Augmented Dickey-Fuller Test
----------------------------
This tests the null hypothesis that the residuals show non-stationarity (trends).
Test Statistic         -7.378918e+00
P-Value                 8.574919e-11
# Lags Used    


! WARNING:
 As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
type: <class 'FutureWarning'>, file: d:\anaconda3\envs\exoplanet-tfm\Lib\site-packages\allesfitter\statistics.py, line: 64


'"So say we all." - Battlestar Galactica\n'